In [2]:
import os
import cv2
import numpy as np
from tqdm import tqdm
import shutil

def images_are_identical(img1, img2):
    """
    Check if two images are 100% identical pixel-by-pixel
    Returns True only if every single pixel is exactly the same
    """
    # First check if dimensions match
    if img1.shape != img2.shape:
        return False
    
    # Direct pixel comparison - if any pixel differs, return False
    return np.array_equal(img1, img2)

def find_unique_masks_exact(masks_folder_2100, masks_folder_630, output_folder):
    """
    Find masks in 2100 folder that are not in 630 folder using exact pixel matching
    Only copies masks that are 100% unique
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Load all masks from 630 folder
    print("Loading masks from 630 folder...")
    masks_630 = []
    
    for filename in tqdm(os.listdir(masks_folder_630)):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            img_path = os.path.join(masks_folder_630, filename)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                masks_630.append(img)
    
    print(f"Loaded {len(masks_630)} masks from 630 folder")
    
    # Process masks from 2100 folder
    print("Comparing masks from 2100 folder with 630 folder...")
    unique_count = 0
    duplicate_count = 0
    
    for filename in tqdm(os.listdir(masks_folder_2100)):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            img_path = os.path.join(masks_folder_2100, filename)
            img_2100 = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            
            if img_2100 is None:
                continue
            
            # Check if this exact mask exists in 630 folder
            is_duplicate = False
            for img_630 in masks_630:
                if images_are_identical(img_2100, img_630):
                    is_duplicate = True
                    duplicate_count += 1
                    break
            
            # If not a duplicate, copy to output folder with original filename
            if not is_duplicate:
                output_path = os.path.join(output_folder, filename)
                shutil.copy2(img_path, output_path)
                unique_count += 1
    
    print(f"Results:")
    print(f"Total masks in 2100 folder: {len(os.listdir(masks_folder_2100))}")
    print(f"Duplicate masks found: {duplicate_count}")
    print(f"Unique masks copied: {unique_count}")
    print(f"Output folder: {output_folder}")
    
    return unique_count

# Configuration - Update these paths with your actual folders
masks_2100_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_masks"
masks_630_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_masks"
output_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_test_masks"

# Run the function
unique_count = find_unique_masks_exact(masks_2100_folder, masks_630_folder, output_folder)

Loading masks from 630 folder...



00%|███████████████████████████████████████| 630/630 [00:00<00:00, 1441.39it/s]

Loaded 630 masks from 630 folder
Comparing masks from 2100 folder with 630 folder...


100%|██████████████████████████████████████| 2100/2100 [00:11<00:00, 179.16it/s]

Results:
Total masks in 2100 folder: 2100
Duplicate masks found: 781
Unique masks copied: 1319
Output folder: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_test_masks


In [3]:
import os
import shutil
from tqdm import tqdm

def create_corresponding_images_folder(output_masks_folder, images_2100_folder, output_images_folder):
    """
    Create a folder with images corresponding to the masks in output_masks_folder
    Images are copied from images_2100_folder with the same filenames
    """
    # Create output images folder if it doesn't exist
    os.makedirs(output_images_folder, exist_ok=True)
    
    print("Creating corresponding images folder...")
    
    # Get all mask filenames from output masks folder
    mask_filenames = [f for f in os.listdir(output_masks_folder) 
                     if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
    
    print(f"Found {len(mask_filenames)} masks in output folder")
    
    # Copy corresponding images
    copied_count = 0
    missing_count = 0
    
    for mask_filename in tqdm(mask_filenames):
        # Construct source image path
        source_image_path = os.path.join(images_2100_folder, mask_filename)
        # Construct destination image path
        dest_image_path = os.path.join(output_images_folder, mask_filename)
        
        # Check if image exists and copy it
        if os.path.exists(source_image_path):
            shutil.copy2(source_image_path, dest_image_path)
            copied_count += 1
        else:
            print(f"Warning: Image not found: {source_image_path}")
            missing_count += 1
    
    print(f"Results:")
    print(f"Successfully copied {copied_count} images")
    print(f"Missing images: {missing_count}")
    print(f"Output images folder: {output_images_folder}")
    
    return copied_count

# Configuration - Update these paths
output_masks_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_test_masks"  # Folder we created earlier
images_2100_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_images"   # Folder with original 2100 images
output_images_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_test_images" # New folder for corresponding images

# Run the function
copied_count = create_corresponding_images_folder(output_masks_folder, images_2100_folder, output_images_folder)

Creating corresponding images folder...
Found 1319 masks in output folder


100%|██████████████████████████████████████| 1319/1319 [00:07<00:00, 184.85it/s]

Results:
Successfully copied 1319 images
Missing images: 0
Output images folder: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_test_images


In [5]:
import os
import shutil
from tqdm import tqdm

def create_corresponding_images_folder(output_masks_folder, images_2100_folder, output_images_folder):
    """
    Create a folder with images corresponding to the masks in output_masks_folder
    Images are copied from images_2100_folder with the same filenames
    """
    # Create output images folder if it doesn't exist
    os.makedirs(output_images_folder, exist_ok=True)
    
    print("Creating corresponding images folder...")
    
    # Get all mask filenames from output masks folder
    mask_filenames = [f for f in os.listdir(output_masks_folder) 
                     if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
    
    print(f"Found {len(mask_filenames)} masks in output folder")
    
    # Copy corresponding images
    copied_count = 0
    missing_count = 0
    
    for mask_filename in tqdm(mask_filenames):
        # Construct source image path
        source_image_path = os.path.join(images_2100_folder, mask_filename)
        # Construct destination image path
        dest_image_path = os.path.join(output_images_folder, mask_filename)
        
        # Check if image exists and copy it
        if os.path.exists(source_image_path):
            shutil.copy2(source_image_path, dest_image_path)
            copied_count += 1
        else:
            print(f"Warning: Image not found: {source_image_path}")
            missing_count += 1
    
    print(f"Results:")
    print(f"Successfully copied {copied_count} images")
    print(f"Missing images: {missing_count}")
    print(f"Output images folder: {output_images_folder}")
    
    return copied_count

# Configuration - Update these paths
output_masks_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_test_masks"  # Folder we created earlier
images_2100_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_point_masks"   # Folder with original 2100 images
output_images_folder = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_test_point_masks" # New folder for corresponding images

# Run the function
copied_count = create_corresponding_images_folder(output_masks_folder, images_2100_folder, output_images_folder)

Creating corresponding images folder...
Found 1319 masks in output folder


100%|█████████████████████████████████████| 1319/1319 [00:00<00:00, 2168.75it/s]

Results:
Successfully copied 1319 images
Missing images: 0
Output images folder: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/full_test_point_masks
